In [ ]:
from sudoku_grover import *
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
import numpy as np
from qiskit.visualization import plot_histogram

In [ ]:
sudoku = generate_random_puzzle()

print_sudoku_board(sudoku, f"Rätsel (Eingabe)")

# 1. Analyse des Suchraums
candidates = find_cell_candidates(sudoku)
constraints = map_constraints_to_qubits(sudoku)

print("Kandidaten pro Zelle (Ergebnis des Preprogressing): ", candidates, "\n")
print("Vergleichsbedingungen (Leere Zelle zeilenweise nummerriert von oben links beginnend): ", [[int(a), int(b)] for a,b in constraints], "\n")

# N = Größe des Suchraums (Produkt der Möglichkeiten pro Zelle)
N_space = np.prod([len(c) for c in candidates])

# 2. Bestimme M (Anzahl der gültigen Lösungen) klassisch für die Adaption
M_solutions = count_solutions(sudoku, candidates)

# 3. Berechne optimale Iterationen
# Formel: pi/4 * sqrt(N/M)
if N_space > M_solutions:
    optimal_iterations = np.pi / (4*np.arcsin(np.sqrt(M_solutions/N_space))) - 0.5
    k = round(optimal_iterations)
else:
    optimal_iterations = 1 # Sofortige Messung, da fast alles eine Lösung ist
    k = 1

print(f"Suchraum N: {N_space}, Lösungen M: {M_solutions}")
print(f"Grover-Iterationen: {k}")
print(f"Optimale Grover-Iterationen: {optimal_iterations}\n")

# --- Quantenschaltkreis Aufbau ---
qr = QuantumRegister(29)
cr = ClassicalRegister(16)
qc = QuantumCircuit(qr, cr)

prepare_initial_state(qc, qr, sudoku)
qc.x(qr[28])
qc.h(qr[28])

for _ in range(k):
    sudoku_oracle(qc, qr, constraints)
    diffusion(qc, qr, sudoku)

qc.measure(qr[0:16], cr[0:16])

# Simulation
backend = AerSimulator(method='matrix_product_state')
transpiled_qc = transpile(qc, backend)
job = backend.run(transpiled_qc, shots=1000)
result = job.result().get_counts()

# Auswertung
highest_res = max(result, key=result.get)
solution_vals = quantum_state_to_sudoku_format(highest_res)

solved_sudoku = sudoku.copy()
solved_sudoku[sudoku == 0] = [int(d) for d in solution_vals]

# highest_res[i:i-2:-1]
highest_result = [highest_res[i] + highest_res[i-1] for i in range(15, 0, -2)]
solution_vals = [i+ " " for i in solution_vals]
print("Häufigster gemessener Zustand: ", highest_res, f"\n", " Q-Bit (q_0q_1, q_2q_3, ...): ",  highest_result, f"\n"," Zahlen:                      ",  solution_vals)

if valid_sudoku(solved_sudoku):
    print_sudoku_board(solved_sudoku, "Lösung GEFUNDEN ✅")
else:
    print_sudoku_board(solved_sudoku, "Lösung FALSCH ❌")

plot_histogram(result,number_to_keep=8)